# Лабораторная работа № 5 — запросы индивидуального задания (2.1–2.12)
**Вариант 11 «Автостоянка», база данных `parking`**

Запросы 2.1–2.12 к базе `parking`: соединения всех типов, DISTINCT, BETWEEN, LIMIT, OFFSET, LIKE, POSIX, агрегаты, вложенный запрос, CASE, группировка. Скрипт: `lab5_parking_queries.sql`.

Ноутбук выполняет все запросы по порядку: **Run All** (или `Shift+Enter` по ячейкам). Результат каждого запроса выводится под ячейкой.

Перед запуском: PostgreSQL запущен, база `parking` создана и заполнена (`lab3_parking_tables.sql`, `lab3_parking_data.sql`).
Параметры подключения берутся из переменных окружения `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD`, `PGDATABASE`; по умолчанию `127.0.0.1:5432`, пользователь — текущий, база `parking`.

Рабочий формат проекта — Jupyter. Зависимости: `pip install "psycopg[binary]" pandas ipykernel`; поддерживается и psycopg2. Подготовка БД: `lab2_parking_db.ipynb`, затем `lab3_parking.ipynb`.

[Все лабораторные](../../README.md) · [Отчёты ЛР5](../../labs/lab05/)

In [1]:
import os
import getpass
import pandas as pd
from IPython.display import display
try:
    import psycopg
except ImportError:
    import psycopg2 as psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)
# При необходимости задайте PGHOST, PGPORT, PGUSER, PGPASSWORD перед запуском.
# Пароль в ноутбуке не сохраняйте: драйвер также поддерживает ~/.pgpass.
DB_NAME = os.getenv("PGDATABASE", "parking")
conn = psycopg.connect(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", getpass.getuser()),
    password=os.getenv("PGPASSWORD") or None,
    dbname=DB_NAME,
)
conn.autocommit = False

def q(sql):
    """Выполнить инструкцию; выборку показать таблицей."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            if cur.description:
                frame = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
                frame.index = range(1, len(frame) + 1)
                display(frame)
                print(f"Строк: {len(frame)}")
            else:
                print(cur.statusmessage)
    except Exception:
        conn.rollback()
        raise

run = q
print("Подключение:", DB_NAME)


Подключение: parking


### 2.1. Все типы соединений

**Запрос 2.1.1.** INNER JOIN. Вывести договоры с именем клиента, госномером и маркой транспортного средства.

In [2]:
q("""

SELECT c.contr_number AS "Договор",
       cl.cl_name     AS "Клиент",
       v.veh_plate    AS "Госномер",
       v.veh_brand    AS "Марка",
       c.contr_status AS "Статус"
FROM   contracts c
INNER  JOIN clients  cl ON cl.cl_ID  = c.contr_client_ID
INNER  JOIN vehicles v  ON v.veh_ID  = c.contr_veh_ID
ORDER  BY c.contr_ID;

""")

,Договор,Клиент,Госномер,Марка,Статус
1,Д-2024/001,Лебедев Игорь Олегович,А123ВС55,Lada,действующий
2,Д-2024/002,Сафина Марина Рустамовна,В456ЕК55,Kia,действующий
3,Д-2024/003,ООО «Сибтранс»,Е789МН55,ГАЗ,действующий
4,Д-2024/004,Круглов Антон Павлович,К321ОР55,Toyota,действующий
5,Д-2024/005,Нечаева Елена Борисовна,М654СТ55,Hyundai,завершён
6,Д-2024/006,АО «Омскстрой»,Н987УХ55,КамАЗ,действующий
7,Д-2024/007,Тимофеев Роман Андреевич,О147АВ55,Honda,действующий
8,Д-2024/008,Жданова Полина Сергеевна,Р258ЕК55,Renault,расторгнут
9,Д-2024/009,Лебедев Игорь Олегович,С369МН155,BMW,действующий
10,Д-2024/010,Сафина Марина Рустамовна,Т741ОР55,Volkswagen,завершён


Строк: 10


**Запрос 2.1.2.** LEFT JOIN. Вывести все машино-места и договоры на них; места, на которые договоры не оформлялись, выводятся с пустыми значениями.

In [3]:
q("""

SELECT p.pl_sector || '-' || p.pl_number AS "Место",
       p.pl_status                       AS "Статус места",
       c.contr_number                    AS "Договор"
FROM   places p
LEFT   JOIN contracts c ON c.contr_place_ID = p.pl_ID
ORDER  BY p.pl_ID;

""")

,Место,Статус места,Договор
1,A-1,занято,Д-2024/001
2,A-2,занято,Д-2024/002
3,A-3,свободно,Д-2024/010
4,A-4,занято,Д-2024/004
5,A-5,занято,Д-2024/009
6,B-1,занято,Д-2024/003
7,B-2,занято,Д-2024/006
8,B-3,занято,Д-2024/007
9,C-1,свободно,Д-2024/005
10,C-2,свободно,NaN


Строк: 12


**Запрос 2.1.3.** RIGHT JOIN. Вывести тот же список, поменяв таблицы местами: все машино-места находятся справа от слова JOIN.

In [4]:
q("""

SELECT p.pl_sector || '-' || p.pl_number AS "Место",
       p.pl_status                       AS "Статус места",
       c.contr_number                    AS "Договор"
FROM   contracts c
RIGHT  JOIN places p ON c.contr_place_ID = p.pl_ID
ORDER  BY p.pl_ID;

""")

,Место,Статус места,Договор
1,A-1,занято,Д-2024/001
2,A-2,занято,Д-2024/002
3,A-3,свободно,Д-2024/010
4,A-4,занято,Д-2024/004
5,A-5,занято,Д-2024/009
6,B-1,занято,Д-2024/003
7,B-2,занято,Д-2024/006
8,B-3,занято,Д-2024/007
9,C-1,свободно,Д-2024/005
10,C-2,свободно,NaN


Строк: 12


**Запрос 2.1.4.** FULL JOIN. Сопоставить машино-места и действующие договоры: показать места без действующего договора и договоры (завершённые, расторгнутые), которым не соответствует ни одно место при этом условии.

In [5]:
q("""

SELECT p.pl_sector || '-' || p.pl_number AS "Место",
       c.contr_number                    AS "Действующий договор",
       c.contr_status                    AS "Статус"
FROM   places p
FULL   JOIN contracts c
       ON c.contr_place_ID = p.pl_ID AND c.contr_status = 'действующий'
ORDER  BY p.pl_ID, c.contr_ID;

""")

,Место,Действующий договор,Статус
1,A-1,Д-2024/001,действующий
2,A-2,Д-2024/002,действующий
3,A-3,NaN,NaN
4,A-4,Д-2024/004,действующий
5,A-5,Д-2024/009,действующий
6,B-1,Д-2024/003,действующий
7,B-2,Д-2024/006,действующий
8,B-3,Д-2024/007,действующий
9,C-1,NaN,NaN
10,C-2,NaN,NaN


Строк: 15


**Запрос 2.1.5.** CROSS JOIN. Построить все сочетания категорий транспортных средств и периодов тарификации.

In [6]:
q("""

SELECT vc.cat_name  AS "Категория",
       per.tar_period AS "Период"
FROM   vehicle_categories vc
CROSS  JOIN (SELECT DISTINCT tar_period FROM tariffs) per
ORDER  BY vc.cat_ID, per.tar_period;

""")

,Категория,Период
1,Легковой,месяц
2,Легковой,сутки
3,Легковой,час
4,Внедорожник,месяц
5,Внедорожник,сутки
6,Внедорожник,час
7,Грузовой,месяц
8,Грузовой,сутки
9,Грузовой,час
10,Мотоцикл,месяц


Строк: 12


**Запрос 2.1.6.** Рефлексивное соединение (самообъединение). Вывести сотрудников и их непосредственных руководителей.

In [7]:
q("""

SELECT e.emp_name       AS "Сотрудник",
       e.emp_position   AS "Должность",
       ch.emp_name      AS "Руководитель"
FROM   employees e
LEFT   JOIN employees ch ON ch.emp_ID = e.emp_chief_ID
ORDER  BY e.emp_ID;

""")

,Сотрудник,Должность,Руководитель
1,Гордеев Пётр Ильич,администратор,NaN
2,Сомова Анна Львовна,кассир,Гордеев Пётр Ильич
3,Ким Дмитрий Сергеевич,старший кассир,Гордеев Пётр Ильич
4,Ветров Илья Романович,охранник,Гордеев Пётр Ильич
5,Пащенко Ольга Ивановна,охранник,Гордеев Пётр Ильич


Строк: 5


### 2.2. DISTINCT

**Запрос 2.2.** Определить, какие марки транспортных средств хранятся на автостоянке, без повторов.

In [8]:
q("""

SELECT DISTINCT veh_brand AS "Марка"
FROM   vehicles
ORDER  BY 1;

""")

,Марка
1,BMW
2,Honda
3,Hyundai
4,Kia
5,Lada
6,Renault
7,Toyota
8,Volkswagen
9,ГАЗ
10,КамАЗ


Строк: 10


### 2.3. BETWEEN

**Запрос 2.3.** Вывести договоры, оформленные в период с 1 марта по 30 апреля 2024 года включительно.

In [9]:
q("""

SELECT contr_number AS "Договор",
       contr_date   AS "Дата оформления",
       contr_amount AS "Сумма",
       contr_status AS "Статус"
FROM   contracts
WHERE  contr_date BETWEEN '2024-03-01' AND '2024-04-30'
ORDER  BY contr_date;

""")

,Договор,Дата оформления,Сумма,Статус
1,Д-2024/005,2024-03-05,12825.00,завершён
2,Д-2024/006,2024-03-12,91800.00,действующий
3,Д-2024/007,2024-04-02,15000.00,действующий
4,Д-2024/008,2024-04-18,16500.00,расторгнут


Строк: 4


### 2.4. LIMIT

**Запрос 2.4.** Вывести три самые крупные квитанции об оплате.

In [10]:
q("""

SELECT pay_receipt_no AS "Квитанция",
       pay_date       AS "Дата",
       pay_sum        AS "Сумма",
       pay_method     AS "Способ оплаты"
FROM   payments
ORDER  BY pay_sum DESC
LIMIT  3;

""")

,Квитанция,Дата,Сумма,Способ оплаты
1,КВ-000120,2024-02-10 09:05:00,50000.00,перевод
2,КВ-000260,2024-08-12 10:30:00,47200.00,перевод
3,КВ-000162,2024-03-12 16:10:00,45900.00,перевод


Строк: 3


### 2.5. OFFSET

**Запрос 2.5.** Вывести квитанции, занимающие по сумме места с четвёртого по шестое (пропустить три первые).

In [11]:
q("""

SELECT pay_receipt_no AS "Квитанция",
       pay_date       AS "Дата",
       pay_sum        AS "Сумма",
       pay_method     AS "Способ оплаты"
FROM   payments
ORDER  BY pay_sum DESC
LIMIT  3 OFFSET 3;

""")

,Квитанция,Дата,Сумма,Способ оплаты
1,КВ-000131,2024-02-20 14:45:00,33000.00,карта
2,КВ-000205,2024-05-07 15:00:00,13500.00,карта
3,КВ-000112,2024-02-05 11:20:00,13500.00,карта


Строк: 3


### 2.6. LIKE, NOT LIKE

**Запрос 2.6.1.** LIKE. Найти клиентов — юридических лиц, названия которых начинаются с «ООО» или «АО».

In [12]:
q("""

SELECT cl_ID AS "Код", cl_name AS "Клиент", cl_phone AS "Телефон"
FROM   clients
WHERE  cl_name LIKE 'ООО%' OR cl_name LIKE 'АО%'
ORDER  BY cl_ID;

""")

,Код,Клиент,Телефон
1,3,ООО «Сибтранс»,+73812551020
2,6,АО «Омскстрой»,+73812445566


Строк: 2


**Запрос 2.6.2.** NOT LIKE. Найти клиентов, у которых указан адрес электронной почты не на домене mail.ru.

In [13]:
q("""

SELECT cl_ID AS "Код", cl_name AS "Клиент", cl_email AS "Почта"
FROM   clients
WHERE  cl_email NOT LIKE '%@mail.ru'
ORDER  BY cl_ID;

""")

,Код,Клиент,Почта
1,2,Сафина Марина Рустамовна,safina@yandex.ru
2,3,ООО «Сибтранс»,office@sibtrans.ru
3,5,Нечаева Елена Борисовна,necheva@gmail.com
4,6,АО «Омскстрой»,info@omskstroy.ru


Строк: 4


### 2.7. Регулярные выражения POSIX

**Запрос 2.7.1.** Оператор ~. Найти транспортные средства с госномерами региона 55 в формате «буква, три цифры, две буквы, 55» (номера региона 155 не подходят).

In [14]:
q("""

SELECT veh_plate AS "Госномер", veh_brand AS "Марка", veh_model AS "Модель"
FROM   vehicles
WHERE  veh_plate ~ '^[А-Я][0-9]{3}[А-Я]{2}55$'
ORDER  BY veh_ID;

""")

,Госномер,Марка,Модель
1,А123ВС55,Lada,Vesta
2,В456ЕК55,Kia,Rio
3,Е789МН55,ГАЗ,Газель
4,К321ОР55,Toyota,RAV4
5,М654СТ55,Hyundai,Solaris
6,Н987УХ55,КамАЗ,5320
7,О147АВ55,Honda,CB500
8,Р258ЕК55,Renault,Duster
9,Т741ОР55,Volkswagen,Polo


Строк: 9


**Запрос 2.7.2.** Оператор ~* (без учёта регистра). Найти транспортные средства марок Kia, Toyota и Honda.

In [15]:
q("""

SELECT veh_plate AS "Госномер", veh_brand AS "Марка", veh_model AS "Модель"
FROM   vehicles
WHERE  veh_brand ~* '^(kia|toyota|honda)$'
ORDER  BY veh_ID;

""")

,Госномер,Марка,Модель
1,В456ЕК55,Kia,Rio
2,К321ОР55,Toyota,RAV4
3,О147АВ55,Honda,CB500


Строк: 3


### 2.8. Агрегатные функции

**Запрос 2.8.** Получить статистику по квитанциям: количество, общую сумму, среднюю, наименьшую и наибольшую сумму платежа.

In [16]:
q("""

SELECT COUNT(*)                AS "Квитанций",
       SUM(pay_sum)            AS "Общая сумма",
       ROUND(AVG(pay_sum), 2)  AS "Средняя сумма",
       MIN(pay_sum)            AS "Наименьшая",
       MAX(pay_sum)            AS "Наибольшая"
FROM   payments;

""")

,Квитанций,Общая сумма,Средняя сумма,Наименьшая,Наибольшая
1,16,276025.00,17251.56,3500.00,50000.00


Строк: 1


### 2.9. Вложенный запрос

**Запрос 2.9.** Вывести клиентов, у которых есть договор на сумму выше средней суммы всех договоров.

In [17]:
q("""

SELECT cl_ID AS "Код", cl_name AS "Клиент"
FROM   clients
WHERE  cl_ID IN (SELECT contr_client_ID
                 FROM   contracts
                 WHERE  contr_amount > (SELECT AVG(contr_amount)
                                        FROM   contracts))
ORDER  BY cl_ID;

""")

,Код,Клиент
1,3,ООО «Сибтранс»
2,4,Круглов Антон Павлович
3,6,АО «Омскстрой»


Строк: 3


### 2.10. CASE

**Запрос 2.10.** Разделить договоры по размеру суммы на группы «малый» (до 10 000), «средний» (до 50 000) и «крупный».

In [18]:
q("""

SELECT contr_number AS "Договор",
       contr_amount AS "Сумма",
       CASE WHEN contr_amount < 10000 THEN 'малый'
            WHEN contr_amount < 50000 THEN 'средний'
            ELSE 'крупный'
       END          AS "Размер договора"
FROM   contracts
ORDER  BY contr_amount DESC;

""")

,Договор,Сумма,Размер договора
1,Д-2024/003,97200.00,крупный
2,Д-2024/006,91800.00,крупный
3,Д-2024/004,33000.00,средний
4,Д-2024/002,27000.00,средний
5,Д-2024/001,25650.00,средний
6,Д-2024/008,16500.00,средний
7,Д-2024/007,15000.00,средний
8,Д-2024/005,12825.00,средний
9,Д-2024/009,5225.00,малый
10,Д-2024/010,3500.00,малый


Строк: 10


### 2.11. Группировка и простая сортировка

**Запрос 2.11.** Вывести число квитанций и сумму платежей по каждому способу оплаты, отсортировав по убыванию суммы.

In [19]:
q("""

SELECT pay_method  AS "Способ оплаты",
       COUNT(*)    AS "Квитанций",
       SUM(pay_sum) AS "Сумма"
FROM   payments
GROUP  BY pay_method
ORDER  BY SUM(pay_sum) DESC;

""")

,Способ оплаты,Квитанций,Сумма
1,перевод,4,148750.00
2,карта,7,85950.00
3,наличные,5,41325.00


Строк: 3


### 2.12. Группировка с условием и многоуровневая сортировка

**Запрос 2.12.** Вывести клиентов, оформивших не менее двух договоров или на общую сумму более 25 000: число договоров и сумму. Сортировать по убыванию числа договоров, затем по убыванию суммы, затем по имени.

In [20]:
q("""

SELECT cl.cl_name        AS "Клиент",
       COUNT(c.contr_ID) AS "Договоров",
       SUM(c.contr_amount) AS "Общая сумма"
FROM   clients cl
JOIN   contracts c ON c.contr_client_ID = cl.cl_ID
GROUP  BY cl.cl_ID, cl.cl_name
HAVING COUNT(c.contr_ID) >= 2 OR SUM(c.contr_amount) > 25000
ORDER  BY COUNT(c.contr_ID) DESC, SUM(c.contr_amount) DESC, cl.cl_name;

""")

,Клиент,Договоров,Общая сумма
1,Лебедев Игорь Олегович,2,30875.00
2,Сафина Марина Рустамовна,2,30500.00
3,ООО «Сибтранс»,1,97200.00
4,АО «Омскстрой»,1,91800.00
5,Круглов Антон Павлович,1,33000.00


Строк: 5


---
Готово: все запросы выполнены.

In [21]:
conn.rollback()
conn.close()